In [ ]:
from functools import partial

import pandas as pd
from matplotlib import pyplot as plt

from config import PATHS
from config.intervals import SEGMENT
from cycle_extraction.plot_features import plot_features
from feature_extraction.extract_features import FeatureNames
from utils.edf_utils import plot_edf_portion

In [ ]:
feat_names = FeatureNames.ALL_ORDERED
plot_features = partial(plot_features, feat_names=feat_names)

pdir = PATHS.patient_dirs()[8]
print(pdir.name)
segs = pd.read_pickle(pdir.segments_table.pickle).drop(columns=['lead_szr', 'type', 'exists'])
segs = segs[['start_mtz', *feat_names, 'file', 'start_index', ]]
segs.head(10)

In [ ]:
plot_features(segs).show()

In [ ]:
feat = 'var_D'
thresh = 14000
extreme_segs = segs[segs[feat] > thresh]
extreme_segs

In [ ]:
plt.close('all')

segs_around = 2
for seg in extreme_segs.itertuples():
    fig = plot_edf_portion(
        pdir.edf_dir / seg.file,
        start_index=seg.start_index - segs_around * SEGMENT.n_samples,
        n_samples=SEGMENT.n_samples * (segs_around * 2 + 1),
        start_time=seg.start_mtz - SEGMENT.exact_dur * segs_around,
        title=f'T: {seg.start_mtz}\n{feat}: {round(getattr(seg, feat))}\nF: {seg.file}',
        subplot_kw={'ylim': [-500, 500]},
    )

    for ax in fig.axes:
        ax.axvspan(seg.start_mtz, seg.start_mtz + SEGMENT.exact_dur, color='red', alpha=0.2)

    fig.show()